<a href="https://colab.research.google.com/github/afforablewebapp/infosys-maritime-ai/blob/main/Milestone1/Login%20Page.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q streamlit streamlit-option-menu pyngrok pyjwt bcrypt plotly

In [2]:
%%writefile app.py
import secrets
import smtplib
from email.message import EmailMessage
import os, sqlite3, jwt, bcrypt, datetime, time, streamlit as st
import plotly.graph_objects as go
from streamlit_option_menu import option_menu

# 🚀 Force Streamlit Light Theme
os.makedirs(".streamlit", exist_ok=True)
with open(".streamlit/config.toml", "w") as f:
    f.write('[theme]\nbase="light"\nprimaryColor="#ffd803"\nbackgroundColor="#f9fcfc"\nsecondaryBackgroundColor="#e3f6f5"\ntextColor="#2d334a"\n')

st.set_page_config(page_title="Infosys Portal", page_icon="⚡", layout="wide", initial_sidebar_state="expanded")

COLORS = {
    "bg_main": "#f9fcfc", "bg_sidebar": "#e3f6f5", "bg_card": "#ffffff", "bg_card_alt": "#bae8e8",
    "text_main": "#2d334a", "text_heading": "#272343", "text_muted": "#64748b",
    "accent": "#ffd803", "accent_hover": "#e6c300", "accent_text": "#272343",
    "border": "#272343", "border_light": "#bae8e8", "success": "#34d399", "danger": "#f87171"
}

JWT_SECRET = "super-secret-infosys-key-2026"

# 📧 Define Send OTP Email Function
def send_otp_email(receiver_email, otp):
    # Retrieve configurations from environment or user credentials.
    # For Gmail, you would use SMTP server: smtp.gmail.com, Port: 465 (SSL) or 587 (TLS)
    # And your Gmail Address and App Password.
    smtp_server = os.environ.get("SMTP_SERVER", "smtp.gmail.com")
    smtp_port = int(os.environ.get("SMTP_PORT", "465"))
    sender_email = os.environ.get("SENDER_EMAIL", "bhavanagudavalli322@gmail.com")
    sender_password = os.environ.get("SENDER_PASSWORD", "xjeq qfrg ttny edfm")

    # If credentials are not set, print to console as fallback so development is unblocked
    if not sender_email or not sender_password:
        print("="*40)
        print(f"⚠️ SMTP Credentials Missing! Simulated Email to: {receiver_email}")
        print(f"🔑 Simulated OTP Code: {otp}")
        print("="*40)
        # Raise an exception to alert the UI, or simply return for testing
        # Here we raise an exception to guide the user to configure variables, but also allow mock printing
        raise ValueError("SMTP configuration missing. Check backend logs for the simulated OTP.")

    msg = EmailMessage()
    msg["Subject"] = "Your Infosys Portal Password Reset Verification Code"
    msg["From"] = sender_email
    msg["To"] = receiver_email
    msg.set_content(f"""\
Dear User,

You requested a password reset for your Infosys Portal account.
Your one-time verification code is:

{otp}

This OTP is valid for 5 minutes. If you did not request this, please ignore this email.
""")

    # Send the email
    with smtplib.SMTP_SSL(smtp_server, smtp_port) as server:
        server.login(sender_email, sender_password)
        server.send_message(msg)

# 🚀 Neo-Brutalist CSS + Visible Sidebar Re-Open Button
st.markdown("""
<style>
/* Page background */
.stApp {
    background: #f8fafc !important;
}

/* Gradient portal card */
div[data-testid="stAppViewContainer"] .main .block-container {
    background: linear-gradient(
        125deg,
        #003399 0%,
        #1837c5 45%,
        #7028ed 78%,
        #c13df5 100%
    ) !important;
    color: white !important;
    border-radius: 22px;
    padding: 2.5rem;
    margin-top: 2rem;
    max-width: 700px;
}
</style>
""", unsafe_allow_html=True)

def get_db(): return sqlite3.connect("infosys_portal.db", check_same_thread=False)
def hash_txt(t): return bcrypt.hashpw(t.encode(), bcrypt.gensalt()).decode()
def check_txt(t, h): return bcrypt.checkpw(t.encode(), h.encode()) if h else False

with get_db() as conn:
    conn.execute("""CREATE TABLE IF NOT EXISTS users (
        id INTEGER PRIMARY KEY AUTOINCREMENT, username TEXT UNIQUE, email TEXT UNIQUE,
        password_hash TEXT, security_question TEXT, security_answer_hash TEXT)""")
    if not conn.execute("SELECT id FROM users WHERE email='infosys@ai'").fetchone():
        conn.execute("INSERT INTO users VALUES (NULL, ?, ?, ?, ?, ?)",
                     ("Administrator", "infosys@ai", hash_txt("admin@123"), "What is your pet name?", hash_txt("admin")))

def make_jwt(email): return jwt.encode({"email": email, "exp": datetime.datetime.utcnow() + datetime.timedelta(hours=2)}, JWT_SECRET, algorithm="HS256")
def verify_jwt(token):
    try: return jwt.decode(token, JWT_SECRET, algorithms=["HS256"])
    except: return None

for k, v in [("token", None), ("page", "Login"), ("reset_email", None), ("reset_mode", None)]:
    if k not in st.session_state: st.session_state[k] = v

def navigate(p): st.session_state.page = p; st.rerun()

def auth_header(title, sub="Intelligent Analytics Portal"):
    st.markdown(f"""
    <div style="text-align:center;padding:1.5rem 0 1rem;">
        <div style="font-size:40px;margin-bottom:10px;">⚡</div>
        <h1 style="font-size:2rem !important;margin:0;">Infosys Portal</h1>
        <p style="color:{COLORS["text_muted"]};font-size:14px;margin:4px 0 0;">{sub}</p>
    </div>
    <div style="text-align:center;margin-bottom:1.5rem;"><span style="font-size:1.1rem;font-weight:700;color:{COLORS["text_heading"]};">{title}</span></div>
    """, unsafe_allow_html=True)

# ============================================================
# PAGE ROUTING
# ============================================================
if not st.session_state.token:
    if st.session_state.page not in ["Login", "Signup", "Forgot"]:
        st.session_state.page = "Login"

    _, mid, _ = st.columns([1, 1.45, 1])
    with mid:
        if st.session_state.page == "Login":
            auth_header("Sign in to your account")
            email = st.text_input("Email address", placeholder="you@infosys.com").lower().strip()
            pwd = st.text_input("Password", type="password", placeholder="••••••••")
            st.markdown("<br>", unsafe_allow_html=True)

            col_l, col_c, col_r = st.columns([1, 1.15, 1.3])
            if col_l.button("Sign In →", use_container_width=True):
                with get_db() as c: r = c.execute("SELECT password_hash FROM users WHERE email=?", (email,)).fetchone()
                if r and check_txt(pwd, r[0]): st.session_state.token = make_jwt(email); navigate("Dashboard")
                else: st.error("❌ Invalid credentials.")
            if col_c.button("Create Account", use_container_width=True): navigate("Signup")
            if col_r.button("Forgot Password", use_container_width=True): navigate("Forgot")

        elif st.session_state.page == "Signup":
            auth_header("Create an account", "Join Infosys Portal today")
            uname = st.text_input("Full name / Username", placeholder="Jane Doe")
            email = st.text_input("Email address", placeholder="you@infosys.com").lower().strip()
            pwd = st.text_input("Password", type="password", placeholder="Min. 8 characters")
            confirm_pwd = st.text_input("Confirm password", type="password", placeholder="Re-enter password")
            sq = st.selectbox("Security Question", ["What is your pet name?", "What is your mother's maiden name?", "What is your favourite city?"])
            sa = st.text_input("Your answer", placeholder="Security answer")
            st.markdown("<br>", unsafe_allow_html=True)

            if st.button("Create Account & Login →", use_container_width=True):
                if not uname or not email or len(pwd) < 8 or not sa:
                    st.error("⚠️ Please fill all fields (password min 8 chars).")
                elif pwd != confirm_pwd:
                    st.error("❌ Passwords do not match.")
                else:
                    try:
                        with get_db() as c:
                            c.execute("INSERT INTO users VALUES (NULL, ?, ?, ?, ?, ?)", (uname, email, hash_txt(pwd), sq, hash_txt(sa.lower().strip())))
                        st.session_state.token = make_jwt(email)
                        st.success("✅ Account created!")
                        time.sleep(1)
                        navigate("Dashboard")
                    except sqlite3.IntegrityError:
                        st.error("❌ Email or Username already registered.")

            st.markdown("<br>", unsafe_allow_html=True)
            if st.button("← Back to Sign In", use_container_width=True): navigate("Login")

        elif st.session_state.page == "Forgot":
            if st.session_state.get("reset_mode") == "otp":
                auth_header(
                    "Email OTP Verification",
                    "Enter the 6-digit code sent to your registered email"
                )
            elif st.session_state.get("reset_mode") == "sq":
                auth_header(
                    "Security Verification",
                    "Answer your security question to reset your password"
                )
            else:
                auth_header(
                    "Reset your password",
                    "Choose your verification method"
                )

            # STEP 1: Choose verification method
            if not st.session_state.get("reset_email"):
                email = st.text_input(
                    "Registered email address",
                    placeholder="you@infosys.com",
                    key="forgot_email"
                ).strip().lower()

                col_sq, col_otp = st.columns(2)

                if col_sq.button("Via Security Question", use_container_width=True):
                    if not email:
                        st.error("Please enter your registered email address.")
                    else:
                        with get_db() as c:
                            user = c.execute(
                                "SELECT security_question FROM users WHERE email=?",
                                (email,)
                            ).fetchone()

                        if user:
                            st.session_state.reset_email = email
                            st.session_state.sq_p = user[0]
                            st.session_state.reset_mode = "sq"
                            st.rerun()
                        else:
                            st.error("Email not registered. Please enter a registered email.")

                if col_otp.button("Via OTP", use_container_width=True):
                    if not email:
                        st.error("Please enter your registered email address.")
                    else:
                        with get_db() as c:
                            user = c.execute(
                                "SELECT id FROM users WHERE email=?",
                                (email,)
                            ).fetchone()

                        if not user:
                            st.error("Email not registered. Please enter a registered email.")
                        else:
                            otp = str(secrets.randbelow(900000) + 100000)
                            try:
                                send_otp_email(email, otp)
                                st.session_state.reset_email = email
                                st.session_state.reset_mode = "otp"
                                st.session_state.reset_otp = otp
                                st.session_state.reset_otp_expiry = time.time() + 300
                                st.session_state.reset_otp_verified = False
                                st.rerun()
                            except Exception as e:
                                st.error("Unable to send OTP. Check your email settings.")
                                print("OTP email error:", e)

            # STEP 2: OTP verification
            elif st.session_state.get("reset_mode") == "otp":
                st.write(f"OTP sent to: {st.session_state.reset_email}")

                if not st.session_state.get("reset_otp_verified", False):
                    entered_otp = st.text_input(
                        "Enter 6-digit OTP",
                        max_chars=6,
                        placeholder="Enter OTP from your email",
                        key="entered_reset_otp"
                    )

                    if st.button("Verify OTP →", use_container_width=True):
                        saved_otp = st.session_state.get("reset_otp")
                        expiry = st.session_state.get("reset_otp_expiry", 0)

                        if not saved_otp:
                            st.error("Please request a new OTP.")
                        elif time.time() > expiry:
                            st.error("OTP expired. Please request a new OTP.")
                        elif not entered_otp.isdigit() or len(entered_otp) != 6:
                            st.error("Please enter all 6 digits.")
                        elif not secrets.compare_digest(entered_otp, saved_otp):
                            st.error("Incorrect OTP. Please try again.")
                        else:
                            st.session_state.reset_otp_verified = True
                            st.rerun()

                else:
                    st.success("Email verified successfully!")

                    new_password = st.text_input(
                        "New password (minimum 8 characters)",
                        type="password",
                        key="otp_new_password"
                    )
                    confirm_password = st.text_input(
                        "Confirm new password",
                        type="password",
                        key="otp_confirm_password"
                    )

                    if st.button("Reset Password →", use_container_width=True):
                        if time.time() > st.session_state.get("reset_otp_expiry", 0):
                            st.error("OTP expired. Please request a new OTP.")
                        elif len(new_password) < 8:
                            st.error("Password must contain at least 8 characters.")
                        elif new_password != confirm_password:
                            st.error("Passwords do not match.")
                        else:
                            with get_db() as c:
                                c.execute(
                                    "UPDATE users SET password_hash=? WHERE email=?",
                                    (
                                        hash_txt(new_password),
                                        st.session_state.reset_email
                                    )
                                )

                            st.session_state.reset_email = None
                            st.session_state.reset_mode = None
                            st.session_state.reset_otp = None
                            st.session_state.reset_otp_expiry = None
                            st.session_state.reset_otp_verified = False

                            st.success("Password reset successfully!")
                            time.sleep(1)
                            navigate("Login")

            # STEP 3: Security-question verification
            elif st.session_state.get("reset_mode") == "sq":
                st.write("Security question:")
                st.info(st.session_state.get("sq_p", ""))

                answer = st.text_input(
                    "Your security answer",
                    type="password",
                    key="reset_security_answer"
                )
                new_password_sq = st.text_input(
                    "New password (minimum 8 characters)",
                    type="password",
                    key="sq_new_password"
                )
                confirm_password_sq = st.text_input(
                    "Confirm new password",
                    type="password",
                    key="sq_confirm_password"
                )

                if st.button("Reset Password →", use_container_width=True):
                    with get_db() as c:
                        user = c.execute(
                            "SELECT security_answer_hash FROM users WHERE email=?",
                            (st.session_state.reset_email,)
                        ).fetchone()

                    if not user or not check_txt(
                        answer.lower().strip(), user[0]
                    ):
                        st.error("Incorrect security answer.")
                    elif len(new_password_sq) < 8:
                        st.error("Password must contain at least 8 characters.")
                    elif new_password_sq != confirm_password_sq:
                        st.error("Passwords do not match.")
                    else:
                        with get_db() as c:
                            c.execute(
                                "UPDATE users SET password_hash=? WHERE email=?",
                                (
                                    hash_txt(new_password_sq),
                                    st.session_state.reset_email
                                )
                            )

                        st.session_state.reset_email = None
                        st.session_state.reset_mode = None
                        st.success("Password reset successfully!")
                        time.sleep(1)
                        navigate("Login")

            # Cancel button
            st.markdown("<br>", unsafe_allow_html=True)
            if st.button("← Cancel", use_container_width=True):
                st.session_state.reset_email = None
                st.session_state.reset_mode = None
                st.session_state.reset_otp = None
                st.session_state.reset_otp_expiry = None
                st.session_state.reset_otp_verified = False
                st.rerun()

# ============================================================
# DASHBOARDS (ADMIN vs USER)
# ============================================================
else:
    payload = verify_jwt(st.session_state.token)
    if not payload:
        st.session_state.token = None
        st.session_state.page = "Login"
        st.rerun()

    email = payload["email"]
    with get_db() as c: uname = c.execute("SELECT username FROM users WHERE email=?", (email,)).fetchone()[0]

    # 🚀 SIDEBAR MENU
    with st.sidebar:
        st.markdown(f"""
        <div style="padding:16px 8px;text-align:center;">
            <div style="font-size:28px;">⚡</div>
            <div style="font-weight:700;font-size:16px;color:{COLORS["text_heading"]};">Infosys Portal</div>
            <div style="font-size:11px;color:{COLORS["text_muted"]};">{"Admin Panel" if email=="infosys@ai" else "Intelligent Analytics"}</div>
        </div><hr style="border-color:{COLORS["border_light"]};">
        """, unsafe_allow_html=True)

        opts = ["Dashboard", "Settings", "Logout"] if email=="infosys@ai" else ["Dashboard", "Analytics", "Reports", "Logout"]
        menu = option_menu(None, opts, icons=["house", "gear", "box-arrow-right"] if email=="infosys@ai" else ["house", "graph-up", "file-text", "box-arrow-right"],
                           styles={"container": {"background-color": COLORS["bg_sidebar"]}, "nav-link-selected": {"background-color": COLORS["accent"], "color": COLORS["accent_text"]}})
        if menu == "Logout":
            st.session_state.token = None
            st.session_state.page = "Login"
            st.rerun()

    # 🚀 ADMIN DASHBOARD (infosys@ai)
    if email == "infosys@ai":
        st.markdown(f"""
        <div style="background:{COLORS["text_heading"]};border-radius:16px;padding:24px 32px;display:flex;justify-content:space-between;align-items:center;margin-bottom:24px;">
            <div><h1 style="color:{COLORS["accent"]} !important;margin:0;font-size:24px !important;">⚡ Infosys Portal</h1><div style="color:{COLORS["bg_card_alt"]};font-size:13px;">Admin Control Panel</div></div>
            <div style="background:{COLORS["accent"]};padding:8px 18px;border-radius:30px;font-weight:700;color:{COLORS["text_heading"]};">🛡️ {uname}</div>
        </div>
        """, unsafe_allow_html=True)

        st.markdown(f"""
        <div class="pn-card" style="text-align:center;padding:60px 20px;">
            <h1 style="font-size:36px !important;margin-bottom:10px;">🛡️ Admin Dashboard</h1>
            <p style="color:{COLORS["text_muted"]};font-size:16px;font-weight:500;">Welcome to the Administrator area.</p>
        </div>
        """, unsafe_allow_html=True)

    # 🚀 REGULAR USER DASHBOARD
    else:
        st.markdown(f"""
        <div style="background:{COLORS["text_heading"]};border-radius:16px;padding:24px 32px;display:flex;justify-content:space-between;align-items:center;margin-bottom:24px;">
            <div><h1 style="color:{COLORS["accent"]} !important;margin:0;font-size:24px !important;">⚡ Infosys Portal</h1><div style="color:{COLORS["bg_card_alt"]};font-size:13px;">Analytics Dashboard</div></div>
            <div style="background:{COLORS["accent"]};padding:8px 18px;border-radius:30px;font-weight:700;color:{COLORS["text_heading"]};">👤 {uname}</div>
        </div>
        """, unsafe_allow_html=True)

        c1, c2, c3, c4 = st.columns(4)
        for col, icon, lbl, val in [(c1, "📄", "Documents Indexed", "128"), (c2, "🔍", "Searches Today", "47"),
                                    (c3, "📊", "Efficiency Score", "98.4%"), (c4, "🛡️", "Security Status", "Secured")]:
            col.markdown(f"""
            <div class="pn-card" style="text-align:center;">
                <div style="font-size:28px;">{icon}</div>
                <div style="font-size:26px;font-weight:700;color:{COLORS["text_heading"]};">{val}</div>
                <div style="color:{COLORS["text_muted"]};font-size:12px;font-weight:600;">{lbl}</div>
            </div>
            """, unsafe_allow_html=True)

        st.markdown("<br>", unsafe_allow_html=True)
        fig = go.Figure(go.Indicator(mode="gauge+number", value=92, title={"text": "System Health Index", "font": {"color": COLORS["text_heading"], "size": 14}},
                        gauge={"axis": {"range": [0, 100]}, "bar": {"color": COLORS["accent"]}, "bgcolor": COLORS["bg_card_alt"], "borderwidth": 1, "bordercolor": COLORS["border"]}))
        fig.update_layout(paper_bgcolor="rgba(0,0,0,0)", font={"color": COLORS["text_main"], "family": "Inter"}, height=260, margin=dict(l=10, r=10, t=40, b=10))
        st.plotly_chart(fig, use_container_width=True)


Overwriting app.py


In [3]:
import os

log_path = "/content/streamlit.log"
if os.path.exists(log_path):
    print("=== Streamlit Logs (Last 100 Lines) ===")
    with open(log_path, "r") as f:
        lines = f.readlines()
        for line in lines[-100:]:
            print(line, end="")
else:
    print(f"Log file not found at {log_path}. Please check if Streamlit was started correctly and redirecting output to this path.")

=== Streamlit Logs (Last 100 Lines) ===


2026-10-26 05:23:11.238 Port 8501 is not available
                                                                                                                                                                                            Stopping...


In [5]:
import smtplib
from email.message import EmailMessage

def test_smtp_connection():
    smtp_server = "smtp.gmail.com"
    smtp_port = 465
    sender_email = "bhavanagudavalli322@gmail.com"
    sender_password = "xjeq qfrg ttny edfm"
    receiver_email = "bhavanagudavalli322@gmail.com"

    print(f"Initiating connection to {smtp_server}:{smtp_port}...")
    msg = EmailMessage()
    msg["Subject"] = "Infosys Portal SMTP Test"
    msg["From"] = sender_email
    msg["To"] = receiver_email
    msg.set_content("This is a diagnostic message testing your SMTP settings.")

    try:
        with smtplib.SMTP_SSL(smtp_server, smtp_port, timeout=10) as server:
            print("Connected successfully. Attempting login...")
            server.login(sender_email, sender_password)
            print("Login successful! Sending test mail...")
            server.send_message(msg)
            print("🎉 Test email sent successfully! Your settings are correct.")
    except Exception as e:
        print("\n❌ SMTP Connection Failed with Error:")
        print(str(e))
        print("\nTroubleshooting Tips:")
        print("1. Make sure you are using a 16-character 'App Password' from Google, NOT your standard Gmail password.")
        print("2. Make sure 2-Step Verification is enabled on your Gmail account.")

test_smtp_connection()

Initiating connection to smtp.gmail.com:465...
Connected successfully. Attempting login...
Login successful! Sending test mail...
🎉 Test email sent successfully! Your settings are correct.


In [6]:
import os
import time
import subprocess
from pyngrok import ngrok, conf
from google.colab import userdata

# 1. Retrieve your secret token securely from Colab Secrets
NGROK_TOKEN = userdata.get('NGROK_AUTHTOKEN')

# Set the token dynamically in-memory to prevent file lock/overwrite errors
pyngrok_config = conf.PyngrokConfig(auth_token=NGROK_TOKEN)
conf.set_default(pyngrok_config)

# 2. Kill any existing ngrok tunnels or streamlit sessions
ngrok.kill()
!pkill -f streamlit

# 3. Start Streamlit in the background on port 8501
process = subprocess.Popen(
    ["streamlit", "run", "app.py", "--server.port", "8501"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

# 4. Open ngrok tunnel
public_url = ngrok.connect(8501).public_url
print("=" * 60)
print(f"🚀 Infosys Portal Live URL: {public_url}")
print("=" * 60)
print("⌛ App is running! Press [Ctrl + C] or the Colab Stop button to shut down.")

try:
    # Keep the cell active so Ctrl+C can be intercepted
    while True:
        time.sleep(1)
except KeyboardInterrupt:
    print("\n" + "🛑" * 30)
    print("Received Ctrl+C / Stop signal. Shutting down...")
    ngrok.kill()
    process.terminate()
    !pkill -f streamlit
    print("✅ Ngrok tunnel closed and Streamlit server stopped gracefully.")

🚀 Infosys Portal Live URL: https://balcony-parcel-blinking.ngrok-free.dev
⌛ App is running! Press [Ctrl + C] or the Colab Stop button to shut down.

🛑🛑🛑🛑🛑🛑🛑🛑🛑🛑🛑🛑🛑🛑🛑🛑🛑🛑🛑🛑🛑🛑🛑🛑🛑🛑🛑🛑🛑🛑
Received Ctrl+C / Stop signal. Shutting down...
✅ Ngrok tunnel closed and Streamlit server stopped gracefully.


In [4]:
!streamlit run /content/app.py &>/content/streamlit.log &

In [5]:
!pkill -f "streamlit run app.py" || true

^C


In [6]:
!streamlit run app.py --server.port 8501 --server.address 0.0.0.0 > streamlit.log 2>&1 &

In [9]:
import time
import requests

time.sleep(3)
try:
    r = requests.get("http://127.0.0.1:8501/_stcore/health", timeout=5)
    print("Streamlit status:", r.status_code, r.text)
except Exception as e:
    print("Streamlit is not responding:", e)

Streamlit status: 200 ok


In [10]:
!pip -q install pyngrok

ERROR: Operation cancelled by user
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/pip/_internal/cli/base_command.py", line 109, in _run_wrapper
    status = _inner_run()
  File "/usr/local/lib/python3.13/dist-packages/pip/_internal/cli/base_command.py", line 102, in _inner_run
    return self.run(options, args)
           ~~~~~~~~^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.13/dist-packages/pip/_internal/cli/req_command.py", line 106, in wrapper
    return func(self, options, args)
  File "/usr/local/lib/python3.13/dist-packages/pip/_internal/commands/install.py", line 541, in run
    conflicts = self._determine_conflicts(to_install)
  File "/usr/local/lib/python3.13/dist-packages/pip/_internal/commands/install.py", line 662, in _determine_conflicts
    return check_install_conflicts(to_install)
  File "/usr/local/lib/python3.13/dist-packages/pip/_internal/operations/check.py", line 110, in check_install_conflicts
    package_set, _ = create_packag